# Semantic Search RAG Application
This notebook performs semantic search over a video transcript and answers questions using the Groq API and strictly in-memory embeddings.

In [3]:
# Install necessary dependencies
!python3 -m pip install sentence-transformers scikit-learn numpy groq

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [6]:
import os
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from groq import Groq


In [7]:
class TranscriptEmbedder:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        """Initialize the sentence transformer model."""
        self.model = SentenceTransformer(model_name)
    
    def clean_text(self, text: str) -> str:
        """Removes extra spaces and line breaks from the text."""
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def chunk_text(self, text: str, sentences_per_chunk: int = 3) -> list:
        """Splits cleaned text into chunks of roughly `sentences_per_chunk` sentences."""
        text = self.clean_text(text)
        
        # Simple sentence splitting on punctuation
        sentences = re.split(r'(?<=[.!?])\s+', text)
        sentences = [s.strip() for s in sentences if s.strip()]
        
        chunks = []
        for i in range(0, len(sentences), sentences_per_chunk):
            chunk = " ".join(sentences[i:i+sentences_per_chunk])
            chunks.append(chunk)
            
        return chunks

    def embed_chunks(self, chunks: list) -> list:
        """Creates embeddings for a list of text chunks and stores them in memory."""
        if not chunks:
            return []
            
        embeddings = self.model.encode(chunks)
        
        stored_data = []
        for text, emb in zip(chunks, embeddings):
            stored_data.append({
                "text": text,
                "embedding": emb
            })
            
        return stored_data
    
    def embed_query(self, query: str):
        """Creates an embedding for a single user query."""
        return self.model.encode([query])[0]